<a href="https://colab.research.google.com/github/Vengalagagan/NLP/blob/main/2403a52222_NLP_Assignment_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
df = pd.read_csv("/content/spam.csv", encoding='latin-1')
df = df[['v1','v2']]
df.columns = ['label','text']

df['label'] = df['label'].map({'ham':0, 'spam':1})
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['text'] = df['text'].apply(clean_text)

In [23]:
from collections import Counter

words = ' '.join(df['text']).split()
vocab = {word:i+1 for i,(word,_) in enumerate(Counter(words).items())}

def encode(text):
    return [vocab[word] for word in text.split() if word in vocab]

df['encoded'] = df['text'].apply(encode)
max_len = 50

In [25]:
def pad(seq):
    return seq[:max_len] + [0]*(max_len-len(seq))

X = np.array([pad(x) for x in df['encoded']])
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [26]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_data = TextDataset(X_train, y_train)
test_data = TextDataset(X_test, y_test)

In [27]:
class CNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, 100, kernel_size=3)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(100, 2)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0,2,1)
        x = self.pool(torch.relu(self.conv(x)))
        x = torch.max(x, dim=2)[0]
        return self.fc(x)

In [28]:
model = CNNModel(len(vocab)+1, 50)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for X_batch, y_batch in train_data:
        optimizer.zero_grad()
        output = model(X_batch.unsqueeze(0))
        loss = criterion(output, y_batch.unsqueeze(0))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss}")

Epoch 1, Loss: 620.8952512540876
Epoch 2, Loss: 111.67452852857588
Epoch 3, Loss: 35.50509775905357
Epoch 4, Loss: 26.33947461499035
Epoch 5, Loss: 45.928348300911054


In [29]:
y_pred = []
y_true = []

for X_batch, y_batch in test_data:
    output = model(X_batch.unsqueeze(0))
    pred = torch.argmax(output, dim=1)
    y_pred.append(pred.item())
    y_true.append(y_batch.item())

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       964
           1       0.91      0.92      0.92       151

    accuracy                           0.98      1115
   macro avg       0.95      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115

[[951  13]
 [ 12 139]]
